# 台灣醫院加護病床數量抓取工具
**Taiwan Hospital ICU Bed Scraper** — fetches live data from [openinfo.mohw.gov.tw](https://openinfo.mohw.gov.tw) and updates the Excel file.

## How to use
1. Upload `Hospital_Chair_20240731_update.xlsx` to this Colab session (Files → Upload)
2. Run **Cell 1** to install dependencies
3. Run **Cell 2** to test connectivity (3 hospitals)
4. Run **Cell 3** to scrape all 194 hospitals (~3-4 min)
5. Run **Cell 4** to download the updated Excel file

In [ ]:
# Cell 1 — Install dependencies
!pip install -q requests openpyxl beautifulsoup4 lxml
print('✓ Dependencies ready')

In [ ]:
# Cell 2 — Clone the correct branch and set up
import subprocess, os

BRANCH = 'claude/festive-gauss-cANya'

if not os.path.exists('update_icu_beds.py'):
    subprocess.run(['git', 'clone', '--depth=1', '--branch', BRANCH,
                    'https://github.com/apriljeng-collab/ICU.git',
                    '.'], check=True)
    print(f'Repo cloned (branch: {BRANCH})')
else:
    print('Already in repo directory')

# Data quality check and baseline stats
!python update_icu_beds.py --analyze
!python update_icu_beds.py --stats


In [ ]:
# Cell 3 — Test connectivity to MOHW (3 hospitals)
!python update_icu_beds.py --test 3

In [ ]:
# Cell 4 — Run full scrape (194 hospitals, ~3-4 min)
# Progress is cached to icu_results.json — safe to re-run if interrupted
!python update_icu_beds.py

In [ ]:
# Cell 5 — Validate scraped data against baseline, then preview changes
!python update_icu_beds.py --validate
print()
!python update_icu_beds.py --diff


In [ ]:
# Cell 6 — Download the updated Excel file
from google.colab import files
files.download('Hospital_Chair_20240731_update.xlsx')
files.download('icu_results.json')  # cache — useful if you want to re-apply later
print('✓ Files queued for download')

In [ ]:
# Cell 7 (optional) — Download CSV for manual review
!python update_icu_beds.py --export-csv
from google.colab import files
files.download('icu_export.csv')

## Re-applying saved results later
If you have `icu_results.json` from a previous run, upload it and run:
```
!python update_icu_beds.py --from-cache
```